# LangChain Agent Benchmark 01: Models, Prompts, Context, and Outputs

This notebook benchmarks model and prompt variables while keeping the task constant. It uses Hugging Face Inference Providers chat completions.

How to use it: for every model's response, follow the reflections points to give a 1-5 rating for each model.
At the end of the notebook, sum your answers to see how the models fare on these biological tasks.

## 0. Library import and global variable definition

Before running this code cell, make sure you have set up your .env file. This contains your personal credentials (api keys), and is automatically read by `os.getenv()`. 

In [15]:
import os
import time
from typing import Literal

import pandas as pd
from huggingface_hub import InferenceClient

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN")
HF_TIMEOUT = 120
ANSWER_MAX_TOKENS = 400
MODEL_SPECS = [
    {
        "model_type": "good general instruction model",
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "provider": "together",
    },
    # {
    #     "model_type": "biology/medical tuned model",
    #     "model": "m42-health/Llama3-Med42-8B",
    #     "provider": "featherless-ai",
    # },
    # {
    #     "model_type": "tiny weak baseline, not biology-specialized",
    #     "model": "Qwen/Qwen2.5-0.5B-Instruct",
    #     "provider": "featherless-ai",
    # },
]
DEFAULT_MODEL = MODEL_SPECS[0]

if not HF_TOKEN:
    print("Set HUGGINGFACEHUB_API_TOKEN or HF_TOKEN before running the examples.")


## 1. LLM choice

LLM choice affects factual accuracy, biological knowledge, reasoning quality, hallucination behavior, speed, verbosity, and code quality. This is because it relies completely on the content the model has seen during training, somewhat imitating it.


**Reflection Prompts**
- Compare which answers seem most reliable and identify which biological details make you think so.
- Separate content quality from style: is a more fluent answer also more correct?
- Note any omissions, exaggerations, or claims you would want to verify against an external source.


In [17]:
questions = [
    "Explain why batch correction matters in single-cell RNA-seq analysis.",
    # "Explain the role of STAT3 in cancer.",
]
rows = []

for question in questions:
    for spec in MODEL_SPECS:
        start = time.perf_counter()
        answer = None
        error = None
        try:
            # Send one user message to the model.
            messages = [
                {"role": "user", "content": question},
            ]

            # Set up the InferenceClient with the model and provider specified in the spec.
            client = InferenceClient(
                model=spec["model"],
                provider=spec["provider"],
                api_key=HF_TOKEN,
                timeout=HF_TIMEOUT,
            )
            
            # Call the chat_completion method to get the model's response.
            response = client.chat_completion(
                messages=messages,
                # max_tokens=ANSWER_MAX_TOKENS,
                temperature=0,
            )
            answer = response.choices[0].message.content
        except Exception as exc:
            error = repr(exc)

        # Present responses in table format
        rows.append(
            {
                "question": question,
                "model": spec["model"],
                "seconds": round(time.perf_counter() - start, 3),
                "answer": answer,
                "error": error,
            }
        )

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "max-width": "900px"},
)

,question,model,seconds,answer,error
0,Explain why batch correction matters in single-cell RNA-seq analysis.,Qwen/Qwen2.5-7B-Instruct,8.603000,"Batch correction is a crucial step in single-cell RNA-seq (scRNA-seq) analysis because it addresses a common issue in scRNA-seq data: batch effects. Batch effects occur when technical variations unrelated to the biological conditions of interest are introduced during the experimental process. These technical variations can arise from differences in reagents, equipment, or experimental conditions between different batches of samples. Here’s why batch correction is important: 1. **Biological Interpretation**: Batch effects can confound the biological signal, making it difficult to distinguish true biological differences from technical variations. Without batch correction, the analysis might incorrectly identify differences between biological conditions as batch effects, or vice versa. 2. **Consistency and Reproducibility**: Batch effects can lead to inconsistent results across different experiments or batches. This can make it challenging to reproduce findings and compare results from different studies. Batch correction helps ensure that the observed differences are due to the biological conditions being studied rather than technical artifacts. 3. **Integration of Datasets**: In scRNA-seq, it is often necessary to integrate data from multiple batches or experiments to get a comprehensive view of the biological system. Batch effects can make this integration difficult, as the technical differences between batches can mask true biological signals. Batch correction helps align the data from different batches, making integration more straightforward and reliable. 4. **Clustering and Dimensionality Reduction**: Techniques like principal component analysis (PCA) and clustering are commonly used in scRNA-seq to identify groups of cells with similar gene expression patterns. Batch effects can distort these analyses, leading to incorrect clustering and dimensionality reduction. Batch correction ensures that the clustering and dimensionality reduction are based on true biological differences rather than technical artifacts. 5. **Gene Expression Normalization**: Batch effects can also affect the normalization of gene expression levels. Proper normalization is crucial for accurate downstream analysis, such as differential expression analysis. Batch correction helps ensure that gene expression levels are normalized appropriately, leading to more accurate and reliable results. 6. **Biological Insights**: By removing batch effects, researchers can gain clearer insights into the biological processes and cellular states being studied. This can lead to more accurate identification of cell types, cell states, and regulatory networks. In summary, batch correction is essential in scRNA-seq analysis to ensure that the observed differences in gene expression are due to biological factors rather than technical artifacts. This leads to more accurate and reliable biological interpretations, consistent results, and improved integration of datasets.",None


## 2. Temperature

Temperature controls sampling randomness: lower values produce more deterministic answers while higher values increase creativity (can also increase hallucination risk). This works by flattening the output probabilities of the model. 

We show stochasticity by repeating generating the answers multiple times. The key comparison is not only temperature 0 versus 1, but also variation across repeated runs at the same temperature. At temperature 0, repeated answers should be relatively stable; at temperature 1, the selected experiments and framing should vary more.

**Reflection Prompts**
- Compare replicates at the same temperature: which elements remain stable, and which ones change?
- Evaluate whether greater variety produces genuinely more useful ideas or just different wording.
- Decide which temperature you would use for a scientific task and justify the trade-off between creativity and control.


In [ ]:
question = """
The airway bulk RNA-seq experiment identified genes that change expression
after dexamethasone treatment.

Suggest five substantially different follow-up experiments to investigate
the mechanisms behind these changes. Avoid proposing two experiments that
measure the same biological process.
"""
rows = []

for temperature in [0, 0.5, 1.0]:
    for replicate in range(3):
        start = time.perf_counter()

        response = client.chat_completion(
            messages=[{"role": "user", "content": question}],
            temperature=temperature,
            # max_tokens=600,
        )

        rows.append({
            "temperature": temperature,
            "replicate": replicate + 1,
            "seconds": round(time.perf_counter() - start, 3),
            "answer": response.choices[0].message.content,
        })
        
df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


,temperature,replicate,seconds,answer
0,0.000000,1,6.597000,"Certainly! Here are five substantially different follow-up experiments to investigate the mechanisms behind the changes in gene expression after dexamethasone treatment: 1. **Protein-Protein Interaction (PPI) Network Analysis:** - **Objective:** To understand how the changes in gene expression affect protein interactions and pathways. - **Method:** Use the differentially expressed genes (DEGs) to predict and analyze protein-protein interactions using tools like STRING or Cytoscape. This can help identify key regulatory proteins and pathways that are altered by dexamethasone. 2. **Chromatin Immunoprecipitation Sequencing (ChIP-seq):** - **Objective:** To determine the binding sites of transcription factors and other regulatory proteins on the DNA, which can provide insights into the mechanisms of gene regulation. - **Method:** Perform ChIP-seq on key transcription factors or other regulatory proteins that are known to be involved in the response to dexamethasone. This can help identify direct targets and regulatory networks. 3. **RNA Binding Protein (RBP) Immunoprecipitation (RIP-seq):** - **Objective:** To identify RNA-binding proteins that are associated with the mRNAs of differentially expressed genes. - **Method:** Conduct RIP-seq to identify RNA-binding proteins that are enriched in the mRNAs of DEGs. This can help elucidate post-transcriptional regulation mechanisms. 4. **CRISPR-Cas9 Gene Editing:** - **Objective:** To validate the functional importance of specific genes identified in the RNA-seq analysis. - **Method:** Use CRISPR-Cas9 to knockout or knockdown the genes of interest and then re-evaluate the gene expression profile and cellular phenotype. This can help determine the functional impact of these genes on the response to dexamethasone. 5. **Metabolomics Analysis:** - **Objective:** To explore how changes in gene expression affect metabolic pathways and cellular metabolism. - **Method:** Perform metabolomics analysis to identify changes in metabolite levels before and after dexamethasone treatment. This can provide insights into how the altered gene expression impacts cellular metabolism and function. These experiments cover a range of biological processes and can provide complementary information to understand the complex mechanisms behind the changes in gene expression observed after dexamethasone treatment."
1,0.000000,2,6.680000,"Certainly! Here are five substantially different follow-up experiments to investigate the mechanisms behind the changes in gene expression after dexamethasone treatment: 1. **Protein-Protein Interaction (PPI) Network Analysis:** - **Objective:** To understand how the changes in gene expression affect protein interactions and pathways. - **Method:** Use the differentially expressed genes (DEGs) to predict and analyze protein-protein interactions using tools like STRING or Cytoscape. This can help identify key regulatory proteins and pathways that are altered by dexamethasone. 2. **Chromatin Immunoprecipitation Sequencing (ChIP-seq):** - **Objective:** To determine the binding sites of transcription factors and other regulatory proteins on the DNA, which can provide insights into the mechanisms of gene regulation. - **Method:** Perform ChIP-seq on key transcription factors or other regulatory proteins that are known to be involved in the response to dexamethasone. This can help identify direct targets and regulatory networks. 3. **RNA Binding Protein (RBP) Immunoprecipitation (RIP-seq):** - **Objective:** To identify RNA-binding proteins that are differentially associated with mRNAs after dexamethasone treatment. - **Method:** Conduct RIP-seq to identify RNA-binding proteins that are differentially associated with mRNAs. This can help elucidate post-transcriptional regulatory mechanisms. 4. **CRISPR-Cas9 Gene Editing:** - **Objective:** To validate the functional importance of specific genes identified in the RNA-seq analysi

## 3. System prompt

The system prompt sets high-level behavior such as persona, assumption, criteria for tool usage, and any instruction that should always be followed. This is injected at the first turn of the conversation and is kept in memory across all turns. Implemented via `SystemMessage` inside a `ChatPromptTemplate`. 

**Reflection Prompts**
- Observe how the tone changes when the model receives a role or more specific instructions.
- Identify whether the system prompt improves accuracy as well, or mainly changes the form of the answer.
- Ask which instructions make the answer easier to evaluate from a scientific perspective.


In [ ]:
question = "A sample has high mitochondrial RNA and low detected genes. Is it a dying-cell population?"
rows = []

# System prompt: minimal
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": question},
]
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=messages, temperature=0)
rows.append({"system_prompt": "minimal", "answer": response.choices[0].message.content})

# System prompt: biologist persona
messages = [
    {"role": "system", "content": "You are a molecular biologist who explains concepts clearly to computational biology students."},
    {"role": "user", "content": question},
]
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=messages, temperature=0)
rows.append({"system_prompt": "biologist persona", "answer": response.choices[0].message.content})

# System prompt: scientific with strict instructions
messages = [
    {"role": "system", "content": "You are a strict scientific assistant. Separate evidence from speculation, state uncertainty, and avoid unsupported claims."},
    {"role": "user", "content": question},
]
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=messages, temperature=0)
rows.append({"system_prompt": "strict scientific", "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 4. Prompt template

This is a scaffold that controls how the task is framed and allows the injection of dynamic data like user inputs or variables at runtime. Implemented via `PromptTemplate` or `ChatPromptTemplate`. 

**Reflection Prompts**
- Compare how strongly the prompt structure guides the structure of the final answer.
- Notice whether the few-shot examples help the model imitate the format or also reason better.
- Identify which template makes it easiest to correct or compare answers across groups.


In [ ]:
observation = "a T-cell cluster has high interferon-stimulated genes after stimulation"
rows = []

# Free form prompt:  a plain instruction with no examples or required output structure
prompt = f"Explain whether {observation} is biologically meaningful."
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
rows.append({"template": "free form", "answer": response.choices[0].message.content})

# Few-shot prompt: includes an example input-output pair to guide the answer style
prompt = (
    "Example:\n"
    "Observation: high MALAT1 in low-quality nuclei.\n"
    "Answer: This may reflect nuclear RNA content or technical quality; validate with QC metrics and markers.\n\n"
    f"Observation: {observation}\n"
    "Answer:"
)
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
rows.append({"template": "few shot", "answer": response.choices[0].message.content})

# Structured prompt: asks the model to organize its answer into named sections
prompt = (
    f"Observation: {observation}\n"
    "Return sections: Interpretation | Alternative explanations | Checks | Confidence."
)
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
rows.append({"template": "structured", "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 5. Context length

LLM answers depend on what is present in the prompt context window. Models with large context windows receive more background information, which can improve grounding but can also introduce distractions, truncation, and irrelevant details. Context is an intrinsic characteristic of an LLM, and should be taken into account when choosing an LLM for your specific purposes. 


**Reflection Prompts**
- Evaluate which information is preserved when the context is short and which only appears with more context.
- Look for signs of distraction: does more context always make the answer better?
- Discuss what minimum context would be sufficient to answer the question responsibly.


In [ ]:
base_context = "Protocol note: Samples were PBMCs stimulated with IFN-beta for 6 hours. Mitochondrial reads above 20% were filtered."
small_context = base_context
large_context = "\n".join([base_context] + [
    "Marker note: IFIT1, ISG15, MX1, and OAS1 indicate interferon response.",
    "QC note: doublet scores above 0.25 were removed.",
    "Batch note: donor and library chemistry can confound differential expression.",
] * 8)
too_much_context = large_context + "\n" + "\n".join(
    [f"Irrelevant lab inventory line {i}: freezer box metadata unrelated to expression." for i in range(120)]
)

contexts = {"no_context": "", "small_context": small_context, "large_context": large_context, "too_much_context": too_much_context}
question = "Why might IFIT1 and ISG15 be elevated, and what caveats should be checked?"
rows = []
for label, ctx in contexts.items():
    prompt = f"Context:\n{ctx}\n\nQuestion: {question}" if ctx else question
    start = time.perf_counter()
    client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
    response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
    rows.append({"context": label, "seconds": round(time.perf_counter() - start, 3), "answer": response.choices[0].message.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 6. Output parsing

This determines in what format the LLM produces the output, eg. plain text, JSON, pydantic. This is relevant in the context of agents because downstream processes that use LLM's output as input may require it in specific formats. Additionally, structured formats enforce strict validation checking.  

**Reflection Prompts**
- Compare free-form and structured output: which is easier to read, validate, and reuse?
- Observe what is lost when a rich answer has to fit into predefined fields.
- Decide when it is worth enforcing a rigid schema in a scientific pipeline.


In [ ]:
QUESTION = "Interpret high FKBP5 and CRISPLD2 expression in dexamethasone-treated airway smooth muscle cells."


### i. Free text

Free text output is easy for humans to read but unreliable for software to process downstream.

**Reflection Prompts**
- Identify which parts of the answer are clear for humans but difficult to compare automatically.
- Note whether the model signals uncertainty or caveats without being forced to do so by a schema.
- Think about which criteria you would use to assign consistent scores to free-form answers.


In [ ]:
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": QUESTION}], temperature=0)
print(response.choices[0].message.content)


### ii. JSON output

JSON output asks the model to return machine-readable fields. However, malformed JSON can still occur without validation or native structured output.


**Reflection Prompts**
- Check whether the JSON is valid and whether all fields are filled in informatively.
- Compare human readability with usefulness for an automated pipeline.
- Look for examples where the model follows the format but oversimplifies the content.


In [21]:
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser, StrOutputParser

parser = JsonOutputParser()
prompt = f"Return valid JSON only with keys answer, confidence, caveats.\n\nQuestion: {QUESTION}"
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
parser.parse(response.choices[0].message.content)


{'answer': 'High expression of FKBP5 and CRISPLD2 in dexamethasone-treated airway smooth muscle cells suggests potential alterations in stress response and cellular remodeling. FKBP5 is involved in the regulation of glucocorticoid signaling, and its upregulation might indicate a compensatory mechanism to counteract the effects of dexamethasone. CRISPLD2 is associated with cellular degradation processes and may indicate increased cellular stress or damage. However, the specific biological implications require further investigation.',
 'confidence': 'Moderate',
 'caveats': 'The interpretation is based on existing literature and may not fully capture the complexity of the cellular response. Further experimental validation is recommended.'}

### iii. Pydantic parser

Pydantic parser validates types and required fields, making outputs usable directly in downstream Python code.

In this example, we can access the `parser.answer`, `parser.confidence`, and `parser.caveats` variables within Python, making it much easier to build reliable pipelines and AI agents. 


**Reflection Prompts**
- Observe which schema constraints help catch errors or ambiguity.
- Evaluate whether validation improves scientific quality or only formal compliance.
- Discuss which fields you would add to make the answer more useful in a lab setting.


In [24]:
from pydantic import BaseModel, Field, ValidationError

class BioAnswer(BaseModel):
    answer: str
    confidence: Literal["low", "medium", "high"]
    caveats: list[str]

parser = PydanticOutputParser(pydantic_object=BioAnswer)

prompt = f"""
You are assisting with RNA-seq interpretation.
{parser.get_format_instructions()}
Question:
{QUESTION}
"""

client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
response = parser.parse(response.choices[0].message.content)
print(response)


answer='High expression of FKBP5 and CRISPLD2 in dexamethasone-treated airway smooth muscle cells suggests a potential response to glucocorticoid treatment. FKBP5 is known to modulate the effects of glucocorticoids, and its upregulation may indicate a heightened sensitivity to dexamethasone. CRISPLD2, while less well-known in the context of glucocorticoid signaling, may be involved in cellular stress responses or inflammation, which are often modulated by dexamethasone treatment.' confidence='medium' caveats=['The interpretation of these findings is based on the known functions of these genes in other contexts, and further experimental validation is needed to confirm their specific roles in dexamethasone-treated airway smooth muscle cells.', 'The high expression could also be due to other factors such as cellular stress or inflammation, which are common in airway smooth muscle cells under various conditions.']


Downstream use of pydantic parsing: routing the pipeline/agent based on confidence:

In [25]:
if response.confidence == "high":
    print("✅ Proceed with pathway enrichment.")
elif response.confidence == "medium":
    print("⚠️ Check supporting literature before interpreting.")
else:
    print("❌ Collect more evidence before drawing conclusions.")

⚠️ Check supporting literature before interpreting.


### vi. Markdown formatting

Markdown formatting controls the shape of the answer, such as plain text, tables, or bullet lists, which affects readability and downstream copy-paste usability.


**Reflection Prompts**
- Compare which format makes it easiest to quickly find results, caveats, and conclusions.
- Evaluate whether the table forces a more orderly comparison than the paragraph or bullet points.
- Ask which format you would use for personal notes, scientific reports, or machine-readable output.


In [26]:
formats = {
    "plain_text": "Answer in one short paragraph.",
    "table": "Answer as a markdown table with columns Finding, Interpretation, Caveat.",
    "bullet_list": "Answer as concise bullet points.",
}
question = "Summarize how to interpret marker genes, QC metrics, and batch effects in scRNA-seq."
rows = []
for label, instruction in formats.items():
    prompt = f"{instruction}\n\n{question}"
    client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
    response = client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0)
    rows.append({"format": label, "answer": response.choices[0].message.content})

for row in rows:
    print("\n###", row["format"], "\n", row["answer"])



### plain_text 
 In single-cell RNA sequencing (scRNA-seq), marker genes are used to identify and characterize cell types based on their unique gene expression patterns. QC metrics, such as the number of detected genes and total counts, help filter out low-quality cells to ensure reliable analysis. Batch effects, which can introduce technical variability between different sequencing runs or samples, need to be corrected to avoid false biological conclusions. Proper normalization and integration techniques are crucial for handling batch effects and ensuring the accurate interpretation of scRNA-seq data.

### table 
 | Finding                | Interpretation                                                                 | Caveat                                                                 |
|------------------------|-------------------------------------------------------------------------------|------------------------------------------------------------------------|
| Marker Genes 

## 7. Streaming

Streaming controls whether tokens are delivered incrementally, which affects user experience and perceived latency without necessarily changing the final answer.


**Reflection Prompts**
- Distinguish perceived latency from total time: which matters more for the final user?
- Observe whether seeing the answer as it is generated changes how you evaluate it.
- Think of scientific cases where streaming is useful and cases where it could be distracting.


In [30]:
prompt = "Give a concise explanation of pseudobulk differential expression."

chunks = []
start = time.perf_counter()
client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
for chunk in client.chat_completion(messages=[{"role": "user", "content": prompt}], temperature=0, stream=True):
    if chunk.choices and chunk.choices[0].delta.content:
        text = chunk.choices[0].delta.content
        chunks.append(text)
        print(text, end="")
print("\n\nstream_seconds:", round(time.perf_counter() - start, 3))


Pseudobulk differential expression analysis is a method used in single-cell RNA sequencing (scRNA-seq) data analysis to infer gene expression changes between different conditions or cell types. It involves aggregating gene expression data from multiple single cells into a pseudo-bulk sample, which can then be compared to identify differentially expressed genes. This approach is particularly useful when the number of cells in a specific condition is small, as it increases the statistical power by combining data from similar cells.

stream_seconds: 1.326


## 8. Citation generation

Citation generation asks the model to attach source references to claims, which improves auditability only when the citations are grounded in provided sources or retrieval metadata.


**Reflection Prompts**
- Check whether each important claim is linked to the correct source.
- Distinguish genuinely supported citations from citations added only as decoration.
- Ask which checks would be needed before trusting generated citations.


In [31]:
context = (
    "[S1] IFIT1, IFIT3, ISG15, MX1, and OAS genes are common interferon-stimulated genes.\n"
    "[S2] High mitochondrial RNA fractions can indicate low-quality or stressed cells, but thresholds are tissue and protocol dependent."
)
prompt = f"Use only the sources below and cite each claim with [S1] or [S2].\n\n{context}\n\nQuestion: Interpret high IFIT1 and high mitochondrial RNA."

client = InferenceClient(model=DEFAULT_MODEL["model"], provider=DEFAULT_MODEL["provider"], api_key=HF_TOKEN, timeout=HF_TIMEOUT)
response = client.chat_completion(messages=[{"role": "user", "content": prompt}], max_tokens=ANSWER_MAX_TOKENS, temperature=0)
print(response.choices[0].message.content)


High levels of IFIT1, an interferon-stimulated gene [S1], suggest an active antiviral response or an interferon signaling pathway being activated in the cell. On the other hand, high mitochondrial RNA fractions can indicate low-quality or stressed cells [S2], but it is important to note that thresholds for this interpretation can vary depending on the tissue and the specific protocol used. Therefore, while high IFIT1 expression points towards an interferon-stimulated state, high mitochondrial RNA could suggest cellular stress or low-quality cellular material, with the latter needing context-specific evaluation.
